# Ejemplo de Estimación por Datos Panel

El cuaderno trabaja dos paneles: primero el de **salarios** de Vella y Verbeek (1998), secciones 1 a 8; después el de **inversión de Grunfeld** (1958), sección 9, cuyos resultados se verifican contra los publicados.

Referencia: Vella and M. Verbeek (1998), “Whose Wages Do Unions Raise? A Dynamic Model of Unionism and Wage Rate Determination for Young Men,” Journal of Applied Econometrics 13, 163-183.

## 1. Dependencias

In [ ]:
#!pip install linearmodels==4.24
#!pip install linearmodels==4.5
#!pip install linearmodels

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
from linearmodels.panel.model import PooledOLS, PanelOLS
from linearmodels.panel import RandomEffects
from linearmodels.panel import compare # Para comparar modelos

#
import warnings
warnings.filterwarnings('ignore')

## 2. Importación de datos

In [ ]:
# Importamos el Data Set
data_to_load = 'wage_panel.csv'

# Read the and
wage_df = pd.read_csv(data_to_load)
wage_df.head()

### Los datos importados son:
* nr: person identifier
* year: 1980 to 1987
* black: =1 if black
* exper: labor market experience
* hisp: =1 if Hispanic
* hours: annual hours worked
* married: =1 if married
* educ: years of schooling
* union: =1 if in union
* lwage: log(wage)
* expersq: exper^2
* occupation: Occupation code

In [ ]:
# Adecuaciones al índice para hacerlo Panel:
year = wage_df.year
wage_df = wage_df.set_index(['nr', 'year'])
wage_df#.head()

In [ ]:
# Agregamos una columna adicional de año (opción 1):
wage_df['year'] = pd.Categorical( year )
wage_df.head()

In [ ]:
# Agregamos columnas de dummies de Year (opción 2):
wage_df = pd.concat( [ wage_df , pd.get_dummies(wage_df['year']) ], axis=1)

In [ ]:
# Show data:
wage_df

## 3. Regresión Pooled

In [ ]:
# Definición de variables exógenas y endógena
X = [ 'black','hisp','exper','expersq','married', 'educ', 'union', 'year', 'hours' ]
X = sm.add_constant( wage_df[X] )
X

In [ ]:
# Definición de variables exógenas y endógena
X1 = [ 'black','hisp','exper','expersq','married', 'educ', 'union', 1981, 1982,
       1983, 1984, 1985, 1986, 1987, 'hours' ]
X1 = sm.add_constant( wage_df[X1] )
X1

In [ ]:
#
Y = wage_df[ 'lwage' ]
Y

In [ ]:
# Regresión
model_1 = PooledOLS(Y, X1)
pooled_res_1 = model_1.fit()
print(pooled_res_1)

In [ ]:
# Regresión
model = PooledOLS(Y, X)
pooled_res = model.fit()
print(pooled_res)

## 4. Efectos aleatorios

In [ ]:
# Regresión
model = RandomEffects(Y, X)
re_res = model.fit()
print(re_res)

In [ ]:
# Descomposición de varianza
re_res.variance_decomposition

## 5. Efectos fijos

La celda siguiente **falla a propósito**. Con efectos fijos por individuo, la transformación intragrupos elimina todo lo que no varía en el tiempo: `black`, `hisp` y `educ` (y la constante) desaparecen. Además, `exper` aumenta exactamente un año por año para todos, de modo que queda absorbida por los efectos de año. `linearmodels` se niega a estimar y explica por qué; leer ese mensaje es parte del ejercicio. La celda está envuelta en `try/except` para que el cuaderno se pueda ejecutar completo.

In [ ]:
# Regresion
# Esta celda falla A PROPÓSITO: con efectos fijos por individuo no se pueden estimar las
# variables que no cambian en el tiempo (black, hisp, educ) ni la constante, y 'exper'
# queda absorbida por los efectos de año (crece exactamente un año por año).
# El try/except muestra el mensaje de linearmodels sin detener el cuaderno.
try:
    model = PanelOLS(Y, X, entity_effects = True)
    fe_res = model.fit()
    print(fe_res)
except Exception as error:
    print(type(error).__name__, error)

In [ ]:
# Regresion con efectos fijos por entidad
# Omitimos: 'exper', 'black','hisp', 'educ'
X = [ 'expersq', 'union', 'married', 'year', 'hours' ]
X = sm.add_constant(wage_df[X])
model = PanelOLS(Y, X, entity_effects = True)
fe_res = model.fit()
print(fe_res)

In [ ]:
# Regresion con efectos fijos por entidad y tiempo
# Omitimos: 'exper', 'black','hisp', 'educ', 'year'
X = ['expersq', 'union', 'married', 'hours']
X = sm.add_constant(wage_df[X])
model = PanelOLS(Y, X, entity_effects = True, time_effects = True)
fe_te_res = model.fit()
print(fe_te_res)

## 6. Comparación de modelos

In [ ]:
#
print(compare( { 'Efectos fijos (individuo)': fe_res,
                 'Efectos fijos (individuo y año)': fe_te_res,
                 'Efectos aleatorios': re_res,
                 'Pool': pooled_res } ))

## 7. Varianza Robusta:

In [ ]:
# Regresión
X = ['expersq', 'union', 'married', 'year', 'hours']
X = sm.add_constant(wage_df[X])
model = PanelOLS(Y, X, entity_effects = True)
#fe_res = model.fit(cov_type = 'robust')
# NOTAS: “unadjusted”, “homoskedastic” - Assume residual are homoskedastic, AND
#       “robust”, “heteroskedastic” - Control for heteroskedasticity using White’s estimator
fe_res = model.fit(cov_type = "clustered", cluster_entity = True)
# NOTAS: clust_entity_time = mod.fit(cov_type='clustered', cluster_entity=True, cluster_time=True)

print(fe_res)

## 8. Prueba de Hausman:

In [ ]:
import numpy as np
from scipy import stats

def hausman(fe, re):
    """
    Prueba de Hausman: efectos fijos (consistente) frente a efectos aleatorios
    (eficiente bajo H0). Se comparan sólo los coeficientes que ambos modelos estiman;
    la constante se excluye, porque con efectos fijos no está identificada por separado
    y no forma parte del contraste.
    """
    b = fe.params
    B = re.params

    common_cols = [c for c in b.index.intersection(B.index) if c != 'const']
    b = b[common_cols]
    B = B[common_cols]

    # Diferencia de matrices de covarianza
    v_b = fe.cov[common_cols].loc[common_cols]
    v_B = re.cov[common_cols].loc[common_cols]
    diff_var = v_b - v_B

    # En teoría Var(FE) - Var(RE) es definida positiva; en muestras finitas puede no serlo
    min_eig = np.linalg.eigvalsh(diff_var).min()
    if min_eig <= 0:
        print(f"Aviso: Var(FE) - Var(RE) no es definida positiva "
              f"(eigenvalor mínimo {min_eig:.1e}); el estadístico debe leerse con cautela.")

    df = len(common_cols)
    chi2 = float((b - B) @ np.linalg.inv(diff_var) @ (b - B))
    p_val = stats.chi2.sf(chi2, df)

    return chi2, df, p_val


In [ ]:
# 1. Asegurar que las variables sean numéricas/dummies
X_vars = ['expersq', 'union', 'married', 'hours']
X = sm.add_constant(wage_df[X_vars])

# 2. Agregar dummies de año manualmente si 'year' no es parte del índice
year_dummies = pd.get_dummies(wage_df['year'], drop_first=True)
X = pd.concat([X, year_dummies], axis=1)

# Ambos modelos DEBEN usar exactamente la misma X
model_re = RandomEffects(Y, X)
re_res = model_re.fit()

model_fe = PanelOLS(Y, X, entity_effects=True)
fe_res = model_fe.fit()


In [ ]:
# Uso con tus resultados previos:
h_stat, df, p_value = hausman(fe_res, re_res)
print(f"Chi-Squared: {h_stat}, df: {df}, p-value: {p_value}")


**Lectura.** Los grados de libertad son el número de coeficientes comparados: los cuatro regresores que varían dentro del individuo y las siete variables dicotómicas de año. La constante no cuenta. El rechazo es contundente: los efectos individuales están correlacionados con los regresores y efectos aleatorios es inconsistente. El aviso sobre la matriz no definida positiva es la primera de las advertencias prácticas que discute el cap. 5 de las notas; la alternativa robusta es la regresión aumentada de Mundlak.

## 9. Segundo ejemplo: la inversión de Grunfeld

El panel clásico de Grunfeld (1958): inversión bruta anual de grandes empresas estadounidenses entre 1935 y 1954, en función del valor de mercado de la empresa (`value`) y de su acervo de capital (`capital`). Son los mismos datos del cuaderno `SUR_Grunfeld.ipynb` de `Clase_03`: allí se tratan como un sistema de ecuaciones, aquí como un panel.

La versión de `statsmodels` trae 11 empresas. Se usan las 10 de Baltagi (*Econometric Analysis of Panel Data*, cuadro 2.1), sin American Steel, para poder verificar los resultados contra los publicados.

In [ ]:
from statsmodels.datasets import grunfeld

grunfeld_df = grunfeld.load_pandas().data
grunfeld_df = grunfeld_df[grunfeld_df['firm'] != 'American Steel'].copy()
grunfeld_df['year'] = grunfeld_df['year'].astype(int)
grunfeld_df = grunfeld_df.set_index(['firm', 'year'])

Y_g = grunfeld_df['invest']
X_g = sm.add_constant(grunfeld_df[['value', 'capital']])

pool_g = PooledOLS(Y_g, X_g).fit()
fe_g = PanelOLS(Y_g, X_g, entity_effects = True).fit()
re_g = RandomEffects(Y_g, X_g).fit()

print(compare({'Pool': pool_g, 'Efectos fijos': fe_g, 'Efectos aleatorios': re_g}))

**Verificación.** Los coeficientes reproducen el cuadro 2.1 de Baltagi: pool, 0.1155 y 0.2307; intragrupos, 0.1101 (0.0119) y 0.3101 (0.0174); efectos aleatorios, 0.1098 y 0.3081. Efectos aleatorios queda muy cerca de efectos fijos porque el parámetro $\theta$ de la cuasi-transformación es alto (0.86): con $\theta \to 1$ los dos estimadores coinciden.

Las pruebas para elegir entre los tres modelos —$F$ de efectos fijos, Breusch-Pagan, Hausman y Mundlak— se piden en el ejercicio de aplicación del cap. 5 de las notas.

> Circulan varias versiones de los datos de Grunfeld con errores de transcripción (Kleiber y Zeileis, 2010). Que los resultados coincidan con Baltagi confirma que esta versión es la misma que la suya.